# AI=CAT: Wagahai Novel Generator Pipeline
このノートブックは、元のBashスクリプトで記述された小説生成パイプラインをGoogle Colab環境向けに最適化したものです。
各セルを順番に実行していくことで、データの品質評価からファインチューニングまでを一気通貫で行うことができます。

## パイプライン全体の定数定義と初期設定
まずは処理全体で使用するファイルパスの定義と、環境のセットアップ（Phase 0）を行います。

In [ ]:
import os
import sys

# パイプライン全体の定数定義
INPUT_CSV = "twnovel_dataset_roseaullus.csv"
CLEANED_CSV = "cleaned_dataset.csv"
NOVELTY_CSV = "analysis_result_with_novelty.csv"
MERGED_DATA = "./wagahai_theme/merged_wagahai_dataset.csv"

print("=========================================")
print("  AI=CAT: Wagahai Novel Generator Pipeline")
print("=========================================")

print("[Phase 0] 環境セットアップ...")
# 00_setup_colab.py を実行
!python 00_setup_colab.py

## Phase 1: データの品質評価とフィルタリング
入力となるCSVファイルが存在するかチェックし、存在すれば評価およびクリーニングのスクリプトを実行します。

In [ ]:
print("[Phase 1] データの品質評価とフィルタリング...")

if not os.path.exists(INPUT_CSV):
    print(f"エラー: 入力ファイル {INPUT_CSV} が見つかりません。")
    sys.exit(1)

# クリーニングスクリプトの実行
!python 01_evaluate_and_clean.py --input_file {INPUT_CSV}

## Phase 2: 新奇性分析とクラスタリング
前段階で作成されたクリーンデータを用いて、新奇性の分析とクラスタリングを実行します。

In [ ]:
print("[Phase 2] 新奇性分析とクラスタリング...")

if not os.path.exists(CLEANED_CSV):
    print(f"エラー: クリーンデータ {CLEANED_CSV} が見つかりません。")
    sys.exit(1)

# 新奇性分析スクリプトの実行
!python 02_novelty_analysis.py --input_file {CLEANED_CSV}

## Phase 3: 学習用データの整形と分割
分析結果のCSVを元に、機械学習モデルのトレーニングに適した形式への変換とデータの分割を行います。

In [ ]:
print("[Phase 3] 学習用データの整形と分割...")

if not os.path.exists(NOVELTY_CSV):
    print(f"エラー: 新奇性分析結果 {NOVELTY_CSV} が見つかりません。")
    sys.exit(1)

# トレーニングデータ準備スクリプトの実行
!python 03_prepare_training.py --input_file {NOVELTY_CSV}

## Phase 4: 「吾輩はAIである」テーマのデータ強化
特定のテーマ（吾輩はAIである）に沿ったデータの水増しや拡張、マージ処理を行います。

In [ ]:
print("[Phase 4] '吾輩はAIである' テーマのデータ強化...")

if not os.path.exists(NOVELTY_CSV):
    print(f"エラー: 新奇性分析結果 {NOVELTY_CSV} が見つかりません。")
    sys.exit(1)

# データ強化スクリプトの実行
!python 04_setup_wagahai.py --input_file {NOVELTY_CSV} --merge

## Phase 5: ファインチューニング（学習の実行）
Phase 4 でマージされた専用データが存在する場合はそれを使用し、存在しない場合はPhase 3の標準的なデータを使用してファインチューニングを実行します。

In [ ]:
print("[Phase 5] ファインチューニング開始...")

# 条件に応じたデータパスとエポック数の設定
if os.path.exists(MERGED_DATA):
    print("マージ済みデータを使用して学習します...")
    train_data_path = MERGED_DATA
else:
    print("マージ済みデータが見つからないため、Phase 3 のデータで学習します...")
    train_data_path = "./training_data/train.jsonl"

# トレーニングスクリプトの実行
!python 05_train_wagahai.py --input_data {train_data_path} --output_dir "./wagahai_ai_model" --num_epochs 3

print("=========================================")
print("  パイプライン完了！")
print("  生成モデル: ./wagahai_ai_model/")
print("=========================================")